## Importar librerías y definir rutas

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import os
import re
from sentence_transformers import SentenceTransformer
import torch

# Rutas
INPUT_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/txt_limpio")
METADATA_FOLDER = Path("/home/jupyteruser/work/corpus_upeu/metadatos")
CHUNKS_CSV = METADATA_FOLDER / "chunks.csv"

os.makedirs(METADATA_FOLDER, exist_ok=True)

# Cargar modelo de embeddings multilingüe
# Usamos un modelo balanceado: rápido y eficaz en español
MODEL_NAME = 'paraphrase-multilingual-mpnet-base-v2'
print(f"Cargando modelo {MODEL_NAME}...")
model = SentenceTransformer(MODEL_NAME)
print("Modelo cargado.")

# Verificar dispositivo (CPU/GPU)
device = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(device)
print(f"Usando dispositivo: {device}")

/usr/local/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


Cargando modelo paraphrase-multilingual-mpnet-base-v2...


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Modelo cargado.
Usando dispositivo: cuda


##  Función para dividir texto en chunks con superposición

In [2]:
import re

# >>> Issue 3.2: regex para detectar bloques estructurales <<<
_RE_BLOQUE = re.compile(
    r'(?:^|\n)\s*('
    r'Art[íi]culo\s+\d+[ºo°]?(?:\s*[.-]\s*\d+)?'
    r'|Cap[íi]tulo\s+[IVXLCDM\d]+'
    r'|Secci[óo]n\s+\d+'
    r'|T[íi]tulo\s+[IVXLCDM\d]+'
    r')',
    re.IGNORECASE
)


def chunk_text_estructural(text, tokenizer, max_tokens=300, overlap=50, max_chars_per_chunk=1500):
    """
    Divide texto respetando limites de articulos/capitulos.

    Issues corregidos:
    - max_tokens=200 (en lugar de 400) para evitar prompts > n_ctx (Issue 3.1)
    - Detecta y respeta limites de Articulo/Capitulo/Seccion (Issue 3.2)
    - Tope defensivo de 1500 chars por chunk
    """
    matches = list(_RE_BLOQUE.finditer(text))

    if matches:
        chunks = []
        buffer_cabecera = None
        buffer_contenido = []

        for i, m in enumerate(matches):
            cabecera = m.group(1).strip()
            inicio_contenido = m.end()
            fin_contenido = matches[i+1].start() if i+1 < len(matches) else len(text)
            contenido = text[inicio_contenido:fin_contenido].strip()

            bloque = f"{cabecera}\n{contenido}" if contenido else cabecera

            tokens = tokenizer.encode(bloque, add_special_tokens=False)
            if len(tokens) <= max_tokens:
                if buffer_cabecera is None:
                    buffer_cabecera = cabecera
                    buffer_contenido = [contenido] if contenido else []
                else:
                    buffer_contenido.append(contenido)
                    buffer_completo = f"{buffer_cabecera}\n" + "\n".join(buffer_contenido)
                    if len(tokenizer.encode(buffer_completo, add_special_tokens=False)) > max_tokens:
                        chunks.append(buffer_completo)
                        buffer_cabecera = cabecera
                        buffer_contenido = [contenido] if contenido else []
            else:
                if buffer_cabecera is not None:
                    chunks.append(f"{buffer_cabecera}\n" + "\n".join(buffer_contenido))
                    buffer_cabecera = None
                    buffer_contenido = []
                for j in range(0, len(tokens), max_tokens - overlap):
                    sub = tokenizer.decode(tokens[j:j+max_tokens], skip_special_tokens=True)
                    sub = sub[:max_chars_per_chunk]
                    chunks.append(f"{cabecera}\n{sub}")

        if buffer_cabecera is not None:
            chunks.append(f"{buffer_cabecera}\n" + "\n".join(buffer_contenido))

        return [c for c in chunks if c.strip()]

    else:
        return chunk_text_simple(text, tokenizer, max_tokens, overlap, max_chars_per_chunk)


def chunk_text_simple(text, tokenizer, max_tokens=300, overlap=80, max_chars_per_chunk=1500):
    """Fallback: chunking por tokens sin estructura."""
    tokens = tokenizer.encode(text, add_special_tokens=False)
    chunks = []
    start = 0
    while start < len(tokens):
        end = start + max_tokens
        chunk_tokens = tokens[start:end]
        chunk = tokenizer.decode(chunk_tokens, skip_special_tokens=True)
        if len(chunk) > max_chars_per_chunk:
            chunk = chunk[:max_chars_per_chunk]
        chunks.append(chunk)
        start += (max_tokens - overlap)
    return chunks


## Procesar todos los documentos limpios

In [3]:
txt_files = sorted(INPUT_FOLDER.glob("*.txt"))
print(f"Documentos a procesar: {len(txt_files)}")

all_chunks = []

for txt_path in txt_files:
    doc_name = txt_path.stem  # nombre sin extensión
    print(f"Procesando {doc_name}...")
    
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read()
    
    # Dividir en chunks
    # Usar chunking estructural (respeta articulos/capitulos)
    chunks_texto = chunk_text_estructural(
        text,
        model.tokenizer,
        max_tokens=300,
        overlap=50,
        max_chars_per_chunk=1500,
    )
    print(f"  -> {len(chunks_texto)} chunks generados")
    
    # Para cada chunk, registrar metadatos básicos
    for i, chunk in enumerate(chunks_texto):
        all_chunks.append({
            "documento": doc_name,
            "chunk_id": f"{doc_name}_{i:04d}",
            "texto": chunk,
            "num_tokens": len(model.tokenizer.encode(chunk))
        })

# Crear DataFrame
df_chunks = pd.DataFrame(all_chunks)
print(f"\nTotal de chunks generados: {len(df_chunks)}")
df_chunks.head()

Token indices sequence length is longer than the specified maximum sequence length for this model (5802 > 128). Running this sequence through the model will result in indexing errors


Documentos a procesar: 48
Procesando DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025...
  -> 24 chunks generados
Procesando ESTATUTO 2024. 04-09-2024...
  -> 109 chunks generados
Procesando Guía para la organización y orientación del legajo para la docencia ordinaria...
  -> 35 chunks generados
Procesando MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILLER AUTOMATICO 2020 - 2021...
  -> 16 chunks generados
Procesando Modelo de índice del contenido - Legajo...
  -> 5 chunks generados
Procesando Politica Institucional de Inclusión y diversidad cultural v1...
  -> 3 chunks generados
Procesando Politica Institucional de trabajo digno y protección de la persona v.1...
  -> 3 chunks generados
Procesando Politica-ambiental...
  -> 2 chunks generados
Procesando REGLAMENTO ADMISION 2025.v7...
  -> 101 chunks generados
Procesando REGLAMENTO BECAS 2021 ACTUALIZADO...
  -> 67 chunks generados
Procesando REGLAMENTO CODIGO ETICA INVESTIGACION 2021...
  -> 20 chunks generados
Procesando REGLAMENTO DE EST

,documento,chunk_id,texto,num_tokens
0,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025...,a M DIRECTIVA SOBRE PAGO DE DERECHOS DE PUBLIC...,302
1,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025...,s indexadas de la UPeU en sustitución de la Di...,302
2,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025...,Universidad Peruana Unión. Abog. René Wilberth...,302
3,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025...,"transparencia y uso eficiente de los recursos,...",302
4,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025,DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025...,efectúa únicamente una vez que el manuscrito h...,302


## Generar embeddings para todos los chunks

In [4]:
print("Generando embeddings...")
# La función encode de sentence-transformers acepta una lista de strings y devuelve un array numpy
# Mostramos barra de progreso
from tqdm.auto import tqdm
embeddings = model.encode(df_chunks['texto'].tolist(), 
                          show_progress_bar=True,
                          batch_size=32)  # ajusta batch según tu RAM

# Guardar embeddings como lista de numpy arrays (para luego cargar)
# O guardar en formato numpy
embeddings_list = [emb.tolist() for emb in embeddings]
df_chunks['embedding'] = embeddings_list

print(f"Embeddings generados. Dimensión: {embeddings[0].shape}")

Generando embeddings...


Batches:   0%|          | 0/122 [00:00<?, ?it/s]

Embeddings generados. Dimensión: (768,)


## Guardar DataFrame con chunks y embeddings

In [5]:
# Convertir la columna de embeddings a string para guardar en CSV (no es ideal pero simple)
# Opción mejor: guardar como archivo numpy separado y referenciar desde CSV
# Aquí usaremos un CSV sin los embeddings, y guardaremos los embeddings en .npy

# Guardar DataFrame sin embeddings (solo metadatos)
df_metadata = df_chunks[['documento', 'chunk_id', 'texto', 'num_tokens']]
df_metadata.to_csv(CHUNKS_CSV, index=False, encoding='utf-8')
print(f"Metadatos de chunks guardados en {CHUNKS_CSV}")

# Guardar embeddings como archivo numpy
EMBEDDINGS_NPY = METADATA_FOLDER / "embeddings.npy"
np.save(EMBEDDINGS_NPY, np.array(embeddings, dtype=np.float32))
print(f"Embeddings guardados en {EMBEDDINGS_NPY} (shape: {np.array(embeddings).shape})")

# Guardar también la lista de chunk_ids para correspondencia
CHUNK_IDS_NPY = METADATA_FOLDER / "chunk_ids.npy"
np.save(CHUNK_IDS_NPY, df_chunks['chunk_id'].values)
print(f"Chunk IDs guardados en {CHUNK_IDS_NPY}")

Metadatos de chunks guardados en /home/jupyteruser/work/corpus_upeu/metadatos/chunks.csv
Embeddings guardados en /home/jupyteruser/work/corpus_upeu/metadatos/embeddings.npy (shape: (3886, 768))
Chunk IDs guardados en /home/jupyteruser/work/corpus_upeu/metadatos/chunk_ids.npy


## Resumen del corpus

In [6]:
print("Resumen del corpus chunkificado:")
print(f"  Documentos originales: {len(txt_files)}")
print(f"  Chunks totales: {len(df_chunks)}")
print(f"  Tokens promedio por chunk: {df_chunks['num_tokens'].mean():.1f}")
print(f"  Chunks por documento:")
for doc in sorted(df_chunks['documento'].unique()):
    count = df_chunks[df_chunks['documento'] == doc].shape[0]
    print(f"    - {doc}: {count}")

Resumen del corpus chunkificado:
  Documentos originales: 48
  Chunks totales: 3886
  Tokens promedio por chunk: 285.0
  Chunks por documento:
    - DIRECTIVA SOBRE PUBLICACIONES CIENTÍFICAS 2025: 24
    - ESTATUTO 2024. 04-09-2024: 109
    - Guía para la organización y orientación del legajo para la docencia ordinaria: 35
    - MODIFICACIÓN DIRECTIVA IMPLEMENTACION BACHILLER AUTOMATICO 2020 - 2021: 16
    - Modelo de índice del contenido - Legajo: 5
    - Politica Institucional de Inclusión y diversidad cultural v1: 3
    - Politica Institucional de trabajo digno y protección de la persona v.1: 3
    - Politica-ambiental: 2
    - REGLAMENTO ADMISION 2025.v7: 101
    - REGLAMENTO BECAS 2021 ACTUALIZADO: 67
    - REGLAMENTO CODIGO ETICA INVESTIGACION 2021: 20
    - REGLAMENTO DE ESTUDIOS POSGRADO 2025: 309
    - REGLAMENTO DE ESTUDIOS V5_2025: 366
    - REGLAMENTO DEFENSORIA UNIVERSITARIA 2025 v4: 42
    - REGLAMENTO DOCENCIA ORDINARIA v3.5: 75
    - REGLAMENTO ESTUDIANTE UNIONISTA V3: